In [ ]:
import pickle as pkl
import itertools
import os

import pandas as pd
import numpy as np
import scanpy as sc
import networkx as nx
from scipy import stats
import sklearn.metrics as metrics
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn3
from matplotlib.lines import Line2D

from upsetplot import UpSet, from_memberships
from collections import Counter

plt.rcParams['figure.dpi'] = 300

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

#datasets = ['/BC/CID4513', '/BC/CID4515', '/BC/CID4523', '/BC/CID44971', '/BC/CID44991', '/CRC/SMC01', '/CRC/SMC04', '/CRC/SMC07', '/CRC/SMC14', '/CRC/SMC18', '/CRC/SMC23', '/CRC/SMC25']
'''
all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'],
            'Lee_2020':['SMC01', 'SMC04', 'SMC07', 'SMC09', 'SMC16', 'SMC18'],
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554']}
'''
all_data = {'Wu_2021':['CID4495'],
            'Lee_2020':['SMC01'],
            'Pal_2021':['Triple_negative_0126']}

for paper,datasets in all_data.items():
    for dataset in tqdm(datasets):
        methods=['celloracle']

        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = SAMPLE_PATHS['celloracle'].split('/')[-1]
        print('sample',SAMPLE)

        clones = get_clones(SAMPLE_PATHS['celloracle'])
        print('clones',clones)

        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()

        print(clones)

        boot_grns = {x:{} for x in methods}

        for method in methods:
            boot_grns[method] = {x:{} for x in clones}
            for clone in clones:
                for rate in [0.9]:
                    boot_grns[method][clone][rate] = []
                    for i in range(1, 101):
                        network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                        if method=='celloracle':
                            G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                        if method=='grnboost':
                            thresh = np.histogram(network['importance'])[1][1]
                            G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                        boot_grns[method][clone][rate].append(G)

                jc = []
                for i in range(len(boot_grns[method][clone][rate])-1):
                    jc.append(nx.intersection(boot_grns[method][clone][rate][i], boot_grns[method][clone][rate][i+1]).number_of_edges()/nx.compose(boot_grns[method][clone][rate][i], boot_grns[method][clone][rate][i+1]).number_of_edges())
                #print(method,clone,nx.intersection_all(boot_grns[method][clone][rate]).number_of_edges()/nx.compose_all(boot_grns[method][clone][rate]).number_of_edges())
                print(method,clone,sum(jc)/len(jc))

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

#datasets = ['/BC/CID4513', '/BC/CID4515', '/BC/CID4523', '/BC/CID44971', '/BC/CID44991', '/CRC/SMC01', '/CRC/SMC04', '/CRC/SMC07', '/CRC/SMC14', '/CRC/SMC18', '/CRC/SMC23', '/CRC/SMC25']
'''
all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'],
            'Lee_2020':['SMC01', 'SMC04', 'SMC07', 'SMC09', 'SMC16', 'SMC18'],
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554']}
'''
all_data = {'Wu_2021':['CID4495'],
            'Lee_2020':['SMC01'],
            'Pal_2021':['Triple_negative_0126']}

boot_grns = {}
for paper,datasets in all_data.items():
    for dataset in tqdm(datasets):
        boot_grns[dataset] = {}
        methods=['grnboost','celloracle']
        boot_grns[dataset] = {x:{} for x in methods}
        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = SAMPLE_PATHS['celloracle'].split('/')[-1]
        print('sample',SAMPLE)

        clones = get_clones(SAMPLE_PATHS['celloracle'])
        print('clones',clones)

        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()

        print(clones)
        
        for method in methods:
            boot_grns[dataset][method] = {x:{} for x in clones}
            for clone in clones:
                for rate in [0.9]:
                    boot_grns[dataset][method][clone][rate] = []
                    for i in range(1, 101):
                        network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                        if method=='celloracle':
                            G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                        if method=='grnboost':
                            thresh = np.histogram(network['importance'])[1][1]
                            G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                        boot_grns[dataset][method][clone][rate].append(G)

In [ ]:
dataset_prior = 'Triple_negative_0126'
for paper,datasets in all_data.items():
    for dataset in tqdm(datasets):
        methods=['grnboost','celloracle']
        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = SAMPLE_PATHS['celloracle'].split('/')[-1]

        clones = get_clones(SAMPLE_PATHS['celloracle'])

        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()
        for method in methods:
            for clone in clones:
                jc = []
                for i in range(len(boot_grns[dataset][method][clone][rate])):
                    jc.append(nx.intersection(boot_grns[dataset_prior][method][clone][rate][i], boot_grns[dataset][method][clone][rate][i]).number_of_edges()/nx.compose(boot_grns[dataset_prior][method][clone][rate][i], boot_grns[dataset][method][clone][rate][i]).number_of_edges())
                #print(method,clone,nx.intersection_all(boot_grns[method][clone][rate]).number_of_edges()/nx.compose_all(boot_grns[method][clone][rate]).number_of_edges())
                print(method,clone,sum(jc)/len(jc))
        dataset_prior = str(dataset)

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

#datasets = ['/BC/CID4513', '/BC/CID4515', '/BC/CID4523', '/BC/CID44971', '/BC/CID44991', '/CRC/SMC01', '/CRC/SMC04', '/CRC/SMC07', '/CRC/SMC14', '/CRC/SMC18', '/CRC/SMC23', '/CRC/SMC25']
all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'],
            'Lee_2020':['SMC01', 'SMC04', 'SMC07', 'SMC09', 'SMC16', 'SMC18'],
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554']}

boot_grns = {}

for paper,datasets in all_data.items():
    for dataset in tqdm(datasets):
        boot_grns[dataset] = {}
        methods=['celloracle']
        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}
        SAMPLE = dataset
        print('sample',SAMPLE)
        clones = get_clones(SAMPLE_PATHS['celloracle'])
        print('clones',clones)
        temp = []
        for clone in clones:
            if clone.isdigit():
                temp.append(clone)
        clones = temp.copy()
        print(clones)

        boot_grns[dataset] = {x:{} for x in methods}

        for method in methods:
            boot_grns[dataset][method] = {x:{} for x in clones}
            for clone in clones:
                boot_grns[dataset][method][clone] = []
                for i in range(1, 101):
                    network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                    if method=='celloracle':
                        G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                    if method=='grnboost':
                        thresh = np.histogram(network['importance'])[1][1]
                        G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                    boot_grns[dataset][method][clone].append(G)


In [ ]:
import networkx as nx
import numpy as np
import itertools
import pandas as pd
from scipy.spatial.distance import hamming
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

def compute_edge_presence_vectors(sample_dict):
    """
    sample_dict: {'method1':{'cat1':G,...}, 'method2':{'cat1':G,...} }
    Returns:
        edge_vectors[method][category] = binary numpy vector
        edge_index = list of all edges in sample
    """

    # ---- 1) Collect all edges across ALL methods & categories ----
    all_edges = set()
    for method in sample_dict.values():
        for clone in method.keys():
            for G in method[clone]:
                all_edges |= set(map(tuple, map(sorted, G.edges())))

    all_edges = sorted(all_edges)  # consistent ordering
    edge_index = {e: i for i, e in enumerate(all_edges)}

    # ---- 2) Build binary vectors ----
    edge_vectors = {}

    for method, categories in sample_dict.items():
        edge_vectors[method] = {}
        for cat, G_list in categories.items():
            edge_vectors[method][cat] = {}
            for index in range(len(G_list)):
                G = G_list[index]
                vec = np.zeros(len(all_edges), dtype=int)
                for e in map(tuple, map(sorted, G.edges())):
                    vec[edge_index[e]] = 1
                edge_vectors[method][cat][index] = vec

    return edge_vectors, all_edges


def compute_distances_avg(edge_vectors):
    """
    edge_vectors = {method: {category: binary_vector}}
    Returns:
        intra_all, inter_all:
            lists of averaged intra/inter distances for each method in the sample
    """

    intra_all = []
    inter_all = []

    for method, cats in edge_vectors.items():

        # Sort categories alphabetically (robust for any key names)
        categories = sorted(cats.keys())
        if len(categories) <= 1:
            continue

        # Define N as the last category
        N_cat = categories[-1]
        other_cats = categories[:-1]

        # -------------------------
        # Compute INTRA distances
        # -------------------------
        intra_distances = [
            hamming(cats[c1], cats[c2])
            for c1, c2 in itertools.combinations(other_cats, 2)
        ]

        # Average INTRA distance for this method
        if len(intra_distances) > 0:
            intra_avg = float(np.mean(intra_distances))
            intra_all.append(intra_avg)

        # -------------------------
        # Compute INTER distances
        # -------------------------
        inter_distances = [
            hamming(cats[c], cats[N_cat])
            for c in other_cats
        ]

        # Average INTER distance for this method
        if len(inter_distances) > 0:
            inter_avg = float(np.mean(inter_distances))
            inter_all.append(inter_avg)
        #print(other_cats, intra_distances, inter_distances)

    return intra_all, inter_all

def compute_distances(edge_vectors):
    """
    For a single sample:
    edge_vectors = {method: {category: vec}}
    Returns:
        intra_all, inter_all = combined lists for the sample
    """

    intra_all = []
    inter_all = []

    for method, cats in edge_vectors.items():

        # ---- Sort categories alphabetically (works for any string keys) ----
        categories = sorted(cats.keys())

        if len(categories) <= 1:
            print('WARNING: <=1 clone found')
            continue

        # ---- Intra distances (between all pairs) ----
        for c1, c2 in itertools.combinations(categories, 2):
            for i in range(len(cats[c1])):
                for j in range(len(cats[c2])):
                    d = hamming(cats[c1][i], cats[c2][j])
                    inter_all.append(d)

        # ---- Inter distances (each other category vs N) ----
        for c in categories:
            for i in range(len(cats[c])):
                for j in range(len(cats[c])):
                    d = hamming(cats[c][i], cats[c][j])
                    intra_all.append(d)

    return intra_all, inter_all

def build_intra_inter_for_all_samples(data):
    """
    data: dict of samples → methods → categories → nx graphs
    Returns:
        intra_dict[sample], inter_dict[sample]
    """

    intra_dict = {}
    inter_dict = {}

    for sample, sample_dict in data.items():
        edge_vectors, _ = compute_edge_presence_vectors(sample_dict)
        intra, inter = compute_distances(edge_vectors)

        intra_dict[sample] = intra
        inter_dict[sample] = inter

    return intra_dict, inter_dict



def plot_split_violin(intra_dict, inter_dict, title='Inter vs Intra Category Hamming Distances per Sample'):
    """
    Create split violin plot of intra vs inter distances.
    """

    # Build a dataframe suitable for seaborn
    rows = []
    for sample in intra_dict:
        for v in intra_dict[sample]:
            rows.append({"sample": sample, "distance": v, "type": "intra"})
        for v in inter_dict[sample]:
            rows.append({"sample": sample, "distance": v, "type": "inter"})

    df = pd.DataFrame(rows)
    
    group1 = df[(df['sample'] == 'all samples') & (df['type'] == 'intra')]['distance']
    group2 = df[(df['sample'] == 'all samples') & (df['type'] == 'inter')]['distance']
    print(group1, group2)
    stat, p_value = mannwhitneyu(group1, group2)
    print(title, stat, f"{p_value:e}")

    plt.figure(figsize=(6, 6))
    ax = sns.violinplot(
        x="sample",
        y="distance",
        hue="type",
        data=df,
        split=True,
        inner="quartile",
        palette={'intra':'#029E73','inter':'#D55E00'},
        legend=False
    )
    ax.tick_params(left=True)

    plt.title(title)
    plt.tight_layout()
    plt.show()

temp_dict = boot_grns.copy()
for dataset in boot_grns.keys():
    for method in ['celloracle']:
        for key in temp_dict[dataset]['celloracle'].keys():
            for index in range(len(temp_dict[dataset]['celloracle'][key])):
                temp_dict[dataset][method][key][index].remove_nodes_from(list(nx.isolates(temp_dict[dataset][method][key][index])))

#intra_dict, inter_dict = build_intra_inter_for_all_samples(temp_dict)
#plot_split_violin(intra_dict, inter_dict)


In [ ]:
sns.set_theme(font_scale=1.6)
sns.set_style('white')
#plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if ('CID12345' in key) or ('Triple_negative_BRCA1_0554' in key)]}, {'all samples': [x for key, value in inter_dict.items() for x in value if ('CID12345' in key) or ('Triple_negative_BRCA1_0554' in key)]}, 'Inter vs Intra Clone Hamming Distances all samples TNBC')
plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if ('CID' in key or 'Triple' in key)]}, {'all samples': [x for key, value in inter_dict.items() for x in value if ('CID' in key or 'Triple' in key)]}, 'Inter vs Intra Clone Hamming Distances TNBC')
#plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if ('CID4495' in key)]}, {'all samples': [x for key, value in inter_dict.items() for x in value if ('CID4495' in key)]}, 'Inter vs Intra Clone Hamming Distances CID4495')
#plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if ('CID4523' in key)]}, {'all samples': [x for key, value in inter_dict.items() for x in value if ('CID4523' in key)]}, 'Inter vs Intra Clone Hamming Distances CID4523')
plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if 'SMC' in key]}, {'all samples': [x for key, value in inter_dict.items() for x in value if 'SMC' in key]}, 'Inter vs Intra Clone Hamming Distances all samples CRC')
#plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if 'SMC04' in key]}, {'all samples': [x for key, value in inter_dict.items() for x in value if 'SMC04' in key]}, 'Inter vs Intra Clone Hamming Distances all samples SMC04')
#plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if 'SMC04' in key]}, {'all samples': [x for key, value in inter_dict.items() for x in value if 'SMC04' in key]}, 'Inter vs Intra Clone Hamming Distances SMC04')

## Shuffle experiment

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

#datasets = ['/BC/CID4513', '/BC/CID4515', '/BC/CID4523', '/BC/CID44971', '/BC/CID44991', '/CRC/SMC01', '/CRC/SMC04', '/CRC/SMC07', '/CRC/SMC14', '/CRC/SMC18', '/CRC/SMC23', '/CRC/SMC25']
all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'],
            'Lee_2020':['SMC01', 'SMC04', 'SMC07', 'SMC09', 'SMC16', 'SMC18'],
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554']}

shuffle_grns = {}

for paper,datasets in all_data.items():
    for dataset in tqdm(datasets):
        shuffle_grns[dataset] = {}
        methods=['celloracle']
        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/shuffle_"+method+'/'+paper+'/'+dataset for method in methods}
        SAMPLE = dataset
        print('sample',SAMPLE)
        clones = get_clones(SAMPLE_PATHS['celloracle'])
        print('clones',clones)
        temp = []
        for clone in clones:
            if clone.isdigit():
                temp.append(clone)
        clones = temp.copy()
        print(clones)

        shuffle_grns[dataset] = {x:{} for x in methods}

        for method in methods:
            shuffle_grns[dataset][method] = {x:{} for x in clones}
            for clone in clones:
                shuffle_grns[dataset][method][clone] = {'original':[], 'shuffle':[]}
                network = pd.read_csv(f"{SAMPLE_PATHS[method].replace('shuffle_','bootstrap_')}/CL{clone}-whole-"+method+"-net.tsv", sep='\t')
                if method=='celloracle':
                    G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                if method=='grnboost':
                    thresh = np.histogram(network['importance'])[1][1]
                    G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                shuffle_grns[dataset][method][clone]['original'].append(G)
                for i in range(1, 101):
                    network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                    if method=='celloracle':
                        G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                    if method=='grnboost':
                        thresh = np.histogram(network['importance'])[1][1]
                        G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                    shuffle_grns[dataset][method][clone]['shuffle'].append(G)
                

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

def jaccard_similarity(g1, g2):
    return nx.intersection(g1, g2).number_of_edges()/nx.compose(g1, g2).number_of_edges()


def compute_pairwise_jaccards(shuffle_grns):
    results = {}

    for dataset, dataset_data in shuffle_grns.items():
        print(f"Processing {dataset}")
        results[dataset] = {}

        categories = list(dataset_data["celloracle"].keys())

        # 1) All category pairs
        for cat1, cat2 in itertools.combinations(categories, 2):

            original1 = dataset_data["celloracle"][cat1]["original"]
            original2 = dataset_data["celloracle"][cat2]["original"]

            shuffle1 = dataset_data["celloracle"][cat1]["shuffle"]
            shuffle2 = dataset_data["celloracle"][cat2]["shuffle"]

            n = len(shuffle1)
            m = len(original1)

            if not (len(original2) == len(original1)) or not (len(shuffle1) == len(shuffle2)):
                raise ValueError(f"Length mismatch in {dataset}, {cat1}, {cat2}")

            # 2) Compute pairwise Jaccards
            original_j = [
                jaccard_similarity(original1[i], original2[i])
                for i in range(m)
            ]

            shuffle_j = [
                jaccard_similarity(shuffle1[i], shuffle2[i])
                for i in range(n)
            ]

            pair_key = f"{cat1}-{cat2}"

            results[dataset][pair_key] = {
                "original": original_j,
                "shuffle": shuffle_j
            }

    return results


def plot_results(results):
    #palette = ['#2791F5', '#F5A627', '#4AB300', '#D12C70', '', '']
    
    palette = ['#2f4f4f','#8b4513','#228b22','#000080','#ff0000','#ffd700','#7fff00','#00ffff','#ff00ff','#1e90ff','#ffe4b5','#ff69b4']
    
    stddevs = {'TNBC':[], 'CRC':[]}
    for dataset, dataset_results in results.items():

        #all_shuffle = []
        #all_original = []

        plt.figure(figsize=(7, 5))
        keys = list(dataset_results.keys())
        for pair, pair_data in dataset_results.items():
            #all_shuffle.extend(pair_data["shuffle"])
            #all_original.extend(pair_data["original"])
            ax = sns.histplot(pair_data['shuffle'], stat='proportion', color=palette[keys.index(pair)], alpha=0.6)
            ax.tick_params(left=True, bottom=True)
            plt.axvline(pair_data['original'], linestyle="--", linewidth=2, color=palette[keys.index(pair)])
            if 'CID' in dataset or 'Triple' in dataset:
                stddevs['TNBC'].append(np.abs(np.mean(pair_data['shuffle'])-pair_data['original'][0])/np.std(pair_data['shuffle']))
            else:
                stddevs['CRC'].append(np.abs(np.mean(pair_data['shuffle'])-pair_data['original'][0])/np.std(pair_data['shuffle']))
        #mean_original = np.mean(all_original)
        #ax.set_ylabel('/')
        #ax.set_xlabel('Time')
        #plt.legend(labels=['Original', 'Shuffle'])
        
        plt.title(f"{dataset}")
        plt.xlabel("Jaccard Similarity")
        plt.ylabel("Frequency")

        plt.tight_layout()
        plt.show()
    
    ax = sns.boxplot(stddevs, palette='tab10')
    ax.tick_params(left=True)
    plt.title('Deviation of orginal JC from shuffles')
    plt.ylabel("No. of standard deviations")
    plt.xlabel("Cancer type")
    
    plt.show()
    


# Run everything
#results = compute_pairwise_jaccards(shuffle_grns)
#plot_results(results)


In [ ]:
plot_results(results)

In [ ]:
results['CID44971']

In [ ]:
#g1 = shuffle_grns['TNBC1']['celloracle']['0']['original'][0]
#g2 = shuffle_grns['TNBC1']['celloracle']['0']['original'][1]
nx.intersection(g1, g2).number_of_edges()/nx.compose(g1, g2).number_of_edges()

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt


def jaccard(g1, g2):
    nx.intersection(g1, g2).number_of_edges()/nx.compose(g1, g2).number_of_edges()


def compute_pairwise_jaccards(shuffle_grns):
    results = {}

    for dataset, dataset_data in shuffle_grns.items():
        print(f"Processing dataset: {dataset}")

        results[dataset] = {}
        cat_dict = dataset_data["celloracle"]

        all_shuffle_vals = []
        all_original_vals = []

        for cat in cat_dict.keys():
            key = cat
            results[dataset][key] = {"original": [], "shuffle": []}

            original = cat_dict[cat]["original"]
            shuffle = cat_dict[cat]["shuffle"]
            
            # Compute all pairwise Jaccards (original)
            for g1, g2 in itertools.product(original, original):
                val = jaccard(g1, g2)
                results[dataset][key]["original"].append(val)
                all_original_vals.append(val)

            # Compute all pairwise Jaccards (shuffle)
            for g1, g2 in itertools.product(shuffle, shuffle):
                val = jaccard(g1, g2)
                results[dataset][key]["shuffle"].append(val)
                all_shuffle_vals.append(val)

        # ---- Plotting ----
        plt.figure(figsize=(7, 5))
        sns.histplot(all_shuffle_vals, stat='proportion')
        
        original_mean = np.mean(all_original_vals)
        plt.axvline(original_mean, linewidth=3, color='r')

        plt.title(f"{dataset}")
        plt.xlabel("Jaccard similarity")
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()

    return results


In [ ]:
temp = compute_pairwise_jaccards(shuffle_grns)